In [ ]:
import json
import os

# 1. 파일이 저장된 경로 설정
folder_path = "../data/check_cards_benefits"

# 2. 폴더 내 모든 json 파일 목록 가져오기
file_list = [f for f in os.listdir(folder_path) if f.endswith('.json')]
all_card_data = []

print(f"총 {len(file_list)}개의 파일을 읽어옵니다...")

# 3. 반복문을 돌며 파일 읽기
for file_name in file_list:
    file_path = os.path.join(folder_path, file_name)
    
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
        all_card_data.append(data)

# 결과 확인 (첫 번째 데이터 샘플)
if all_card_data:
    print(f"첫 번째 카드 데이터: {all_card_data[0]['card_name']} ({all_card_data[0]['company']})")

총 371개의 파일을 읽어옵니다...
첫 번째 카드 데이터: 밥바라밥 페이북머니 체크카드 (BC 바로카드)


In [ ]:
documents = []

for card in all_card_data:
    # 비서가 답변을 잘 할 수 있도록 정보를 하나의 문맥(Context)으로 통합
    context = (
        f"카드사: {card['company']}, 카드명: {card['card_name']}. "
        f"주요 혜택: {card['cashback']}, {card['benefit_place']}, {card['discount']}. "
        f"이용 조건: 해외결제 {card['overseas']}, 전월실적 {card['performance']}."
    )
    documents.append(context)

print(f"생성된 검색용 문서 개수: {len(documents)}")

In [13]:
pip install chromadb pandas langchain-community langchain-huggingface

In [10]:
pip install sentence-transformers

In [15]:
import json
import os
import chromadb
from chromadb.utils import embedding_functions

# 1. DB 설정 (로컬 폴더 'card_db'에 저장)
client = chromadb.PersistentClient(path="./card_vector_db")

# 2. 임베딩 모델 설정 (한글 처리에 좋은 다국어 모델 사용)
# OpenAI API 키가 없다면 무료 모델인 HuggingFace를 사용합니다.
emb_func = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="jhgan/ko-sroberta-multitask"
)

# 3. 컬렉션 생성 (이미 있으면 가져옴)
collection = client.get_or_create_collection(
    name="card_info", 
    embedding_function=emb_func
)

# 4. JSON 파일 읽어오기 및 DB 입력
folder_path = "../data/cards"
file_list = [f for f in os.listdir(folder_path) if f.endswith('.json')]

ids = []
documents = []
metadatas = []

for i, file_name in enumerate(file_list):
    with open(os.path.join(folder_path, file_name), 'r', encoding='utf-8') as f:
        card = json.load(f)
        
        # [중요] AI가 검색할 때 참고할 '핵심 문장' 만들기
        # 검색 품질을 결정하는 가장 중요한 단계입니다.
        text_content = (
            f"카드사: {card['company']}, 카드명: {card['card_name']}. "
            f"캐시백 혜택: {card['cashback']}. "
            f"주요 사용처: {card['benefit_place']}. "
            f"기타 할인: {card['discount']}. "
            f"실적 조건: {card['performance']}. "
            f"해외 겸용: {card['overseas']}."
        )
        
        ids.append(f"card_{i}")
        documents.append(text_content)
        # 나중에 특정 카드사만 필터링하고 싶을 때를 대비해 메타데이터 저장
        metadatas.append({
            "company": card['company'],
            "name": card['card_name']
        })

# 5. DB에 한꺼번에 저장
collection.add(
    ids=ids,
    documents=documents,
    metadatas=metadatas
)

print(f"✅ 총 {len(documents)}개의 카드 정보가 Vector DB에 저장되었습니다.")

In [16]:
import chromadb
from chromadb.utils import embedding_functions
import os
import json

# 1. DB 연결
client = chromadb.PersistentClient(path="./card_vector_db")

# 2. 기존 컬렉션이 있다면 삭제 (충돌 방지)
try:
    client.delete_collection(name="card_info")
    print("기존 'card_info' 컬렉션을 삭제했습니다.")
except:
    print("'card_info' 컬렉션이 존재하지 않아 새로 생성합니다.")

# 3. 한국어 임베딩 모델 설정
emb_func = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="jhgan/ko-sroberta-multitask"
)

# 4. 새 컬렉션 생성 (이제 충돌이 나지 않습니다)
collection = client.create_collection(
    name="card_info", 
    embedding_function=emb_func
)

# 5. 데이터 다시 밀어넣기
folder_path = "../data/cards"
file_list = [f for f in os.listdir(folder_path) if f.endswith('.json')]

ids, documents, metadatas = [], [], []

for i, file_name in enumerate(file_list):
    with open(os.path.join(folder_path, file_name), 'r', encoding='utf-8') as f:
        card = json.load(f)
        text_content = (
            f"카드사: {card['company']}, 카드명: {card['card_name']}. "
            f"혜택: {card['cashback']}, {card['benefit_place']}, {card['discount']}. "
            f"실적: {card['performance']}"
        )
        ids.append(f"card_{i}")
        documents.append(text_content)
        metadatas.append({"company": card['company'], "name": card['card_name']})

collection.add(ids=ids, documents=documents, metadatas=metadatas)
print(f"✅ 총 {len(documents)}개의 데이터가 새 임베딩 모델로 저장되었습니다.")

In [17]:
# 사용자의 질문
query = "20대 취업준비생이 교통과 식비에 사용하기 좋은 카드 추천해줘"

# 질문과 가장 유사한 데이터 3개 찾기
results = collection.query(
    query_texts=[query],
    n_results=3
)

print("\n--- 검색 결과 ---")
for i in range(len(results['documents'][0])):
    print(f"순위 {i+1}: {results['documents'][0][i]}")
    print(f"유사도 거리: {results['distances'][0][i]}") # 0에 가까울수록 정확함
    print("-" * 30)

In [ ]:
import os
import chromadb
from chromadb.utils import embedding_functions
from openai import OpenAI

# 1. OpenAI API 키 설정 (본인의 키를 입력하세요)
os.environ["OPENAI_API_KEY"] = "API키 직접 작성할 것" 
client_openai = OpenAI()

# 2. 저장된 Vector DB 불러오기
client_db = chromadb.PersistentClient(path="./card_vector_db")
emb_func = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="jhgan/ko-sroberta-multitask"
)
collection = client_db.get_collection(name="card_info", embedding_function=emb_func)

def ask_card_assistant(user_query):
    # 3. Vector DB에서 관련 정보 검색 (Top 3)
    results = collection.query(
        query_texts=[user_query],
        n_results=3
    )
    
    # 검색된 정보들을 하나의 텍스트로 합침 (Context 생성)
    context = "\n".join(results['documents'][0])
    
    # 4. LLM에게 전달할 프롬프트 작성
    # '지시사항(System Prompt)'을 통해 AI가 수집된 데이터로만 말하도록 제약합니다.
    messages = [
        {"role": "system", "content": f"""
            당신은 카드 고릴라 데이터를 기반으로 하는 친절한 카드 추천 비서입니다.
            반드시 아래 제공된 '카드 정보'만을 바탕으로 답변하세요. 
            만약 정보에 없는 내용이라면 모른다고 정직하게 답하세요.
            
            [카드 정보]
            {context}
        """},
        {"role": "user", "content": user_query}
    ]

    # 5. GPT 답변 생성
    response = client_openai.chat.completions.create(
        model="gpt-4o-mini", # 가성비 좋은 모델 추천
        messages=messages,
        temperature=0.5 # 답변의 일관성을 위해 낮게 설정
    )
    
    return response.choices[0].message.content

# --- 실행 테스트 ---
question = "30대 여자가 편의점에서 제일 사용하기 좋은 카드 소개해줘"
answer = ask_card_assistant(question)

print(f"질문: {question}")
print(f"비서: {answer}")